<a href="https://colab.research.google.com/github/ml4devs/ml4devs-notebooks/blob/master/llm/translate_natural_language_query_to_sql_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1><center>Translate Natural Language Queries to SQL with LLMs: Retrieval Augmented Generation Using GPT and ChromaDB</center></h1>

<p><center>
<address>&copy; Satish Chandra Gupta<br/>
LinkedIn: <a href="https://www.linkedin.com/in/scgupta/">scgupta</a>,
Twitter: <a href="https://twitter.com/scgupta">scgupta</a>
</address>
</center></p>

---

Blog Post: [LLM-Powered Text-to-SQL: RAG Using GPT and ChromaDB](http://www.ml4devs.com/articles/text-to-sql-with-llm-rag-using-gpt-and-chromadb/)

## Introduction

Translating natural language questions into SQL, know as [Text-to-SQL](https://paperswithcode.co/tasks/text-to-sql), has been an [active research problem](https://dblp.org/search?q=text+to+sql) since the early 1970s. Early systems handled narrow domains through hand-crafted grammars and rule-based parsers. Statistical approaches followed in the 1990s and 2000s; deep learning models arrived in the 2010s. Today, large language models (LLMs) can generate SQL for arbitrary schemas without per-database training, yet the core challenge of bridging natural language ambiguity with relational query precision remains challenging in the general case.

This notebook builds a minimal, working end-to-end system: a user asks a question in plain English, the system retrieves the relevant database tables, prompts an LLM to write a SQL query, executes it against a real database, and returns a plain-English answer. The core technique is **Retrieval Augmented Generation (RAG)**. Rather than including the entire database schema in every prompt, we embed table definitions into a vector store and retrieve only the most relevant tables at query time.

### What This Tutorial Covers

- Extracting schema metadata (tables, columns, primary and foreign keys) from a SQLite database
- Representing each table as a `CREATE TABLE` DDL statement and embedding it into a vector store
- Retrieving the most relevant tables for a given natural language query using semantic similarity
- Constructing a prompt from retrieved schemas and asking an LLM to generate SQL
- Executing the query and generating a natural language answer from the results

### How This Tutorial Simplifies Reality

This is an intentionally streamlined example to make the core pattern clear. Production text-to-SQL systems differ in several important ways:

**Retrieval Quality.** This tutorial uses a single lightweight embedding model with cosine similarity search. Real systems combine dense (semantic) search with sparse (keyword/BM25) search, apply cross-encoder re-rankers to reorder candidates, and sometimes run multiple retrieval passes with different query phrasings to improve recall.

**Schema Scale.** The Chinook database has 11 tables. Enterprise databases routinely have hundreds or thousands of tables. At that scale, retrieval precision matters enormously. Selecting the wrong tables produces SQL that references nonexistent columns or misses required joins entirely.

**Query Complexity.** The demo queries here are straightforward aggregations over one or two tables. Real-world queries often involve multi-hop joins, subqueries, window functions, CTEs, and business model definitions ("active customers," "net revenue") that require additional domain knowledge to resolve correctly.

**Schema Annotation.** Column names in practice tend to be terse, abbreviated, or reused across tables. Production systems enrich schemas with natural-language column descriptions, sample values, and business definitions before embedding them, so retrieval and generation both benefit from richer context.

**Validation and Error Recovery.** Generated SQL can be syntactically invalid or logically wrong. Robust systems add a validation loop: parse the SQL, check it with `EXPLAIN`, inspect row counts, and ask the LLM to self-correct on failure before returning a result.

**Security.** Passing an LLM-generated string directly to a database is risky in any context where data is sensitive or the database is writable. Production systems restrict the database user to read-only access and validate queries before execution.

Despite these gaps, the scaffold built here (extract schemas, embed them, retrieve the relevant ones, prompt an LLM, execute, and explain) is the backbone of most production text-to-SQL systems, and a solid foundation for going further.

---

## Setup Environment

### Install Pip Packages

You need Python 3.11 or higher to install [OpenAI Python API library](https://github.com/openai/openai-python).

In [1]:
# You should have Python 3.11 or higher

!python --version


Python 3.12.13


In [2]:
!pip install openai chromadb sentence-transformers python-dotenv SQLAlchemy &> /dev/null


### Upload `.env` File with API Keys

You can either use GPT directly from OpenAI, or you can use Azure OpenAI from Microsoft. You need to create a `.env` file and add the environment variables needed for OpenAI api.

If you are using OpenAI, check your [OpenAI account](https://platform.openai.com/api-keys) for creating API key. Your `.env` file will look like following:

```sh
$ cat .env
OPENAI_API_KEY='sk-YourOpenAiApiKeyHere'
```

If you are using Microsoft Azure OpenAI:
- Go to [Azure Portal](https://portal.azure.com/) > **All Resources**
- Filter the list with Type == Azure OpenAI
- Select the one you plan to use
- If there are none, you can [create and deploy an Azure OpenAI Service resource](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/create-resource?pivots=web-portal#create-a-resource)
- Click on **Keys and Endpoint** on the left menu
- Get `AZURE_OPENAI_API_KEY` and `AZURE_OPENAI_ENDPOINT`
- Next click **Model deployments** on the left menu, and then click **Manage Deployment** button
- Alternatively, you can go to [Azure OpenAI Studio](https://oai.azure.com/), and click **Deployments** on the left menu
- Find (the latest) API version for [Azure OpenAI Service](https://learn.microsoft.com/en-us/azure/ai-services/openai/reference#rest-api-versioning)

Your `.env` file will look like following:
```sh
$ cat .env
AZURE_OPENAI_API_KEY=yourAzureOpenAiApiKey
AZURE_OPENAI_ENDPOINT=https://your-azure-deployment.openai.azure.com/
AZURE_OPENAI_DEPLOYMENT_ID=your-deployment-name
AZURE_OPENAI_API_VERSION=2023-10-01-preview
```

Upload `.env` using Upload File button in Google Colab (or Jupyter Notebook). In worst case scenario, uncomment and modify the relevant lines in the following cell to create `.env` file. Please note that it is dangerous to share such notebooks or check them into git.

In [3]:
# Upload or create a .env file with (Azure) OpenAI API creds

#!echo "OPENAI_API_KEY=sk-YourOpenApiKeyHere" >> .env

#!echo "AZURE_OPENAI_API_KEY=yourAzureOpenAiApiKey" >> .env
#!echo "AZURE_OPENAI_ENDPOINT=https://your-azure-deployment.openai.azure.com/" >> .env
#!echo "AZURE_OPENAI_DEPLOYMENT_ID=your-deployment-name" >> .env
#!echo "AZURE_OPENAI_API_VERSION=2023-10-01-preview" >> .env


### Load `.env` File and Specify (Azure) OpenAI GPT Model

Load environment variables from `.env` file:

In [4]:
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())


Set `IS_AZURE_OPENAI` flag to `True`, if you are using Azure OpenAI:

In [5]:
IS_AZURE_OPENAI: bool = False


Specify model name:

In [6]:
from datetime import datetime

OPENAI_GPT_MODEL_VERSION: str = "gpt-5.4-nano"


---

## Setup OpenAI Client with GPT Model

In [7]:
import json
import os
import openai


In [8]:
def create_open_ai_client():
    if IS_AZURE_OPENAI:
        return openai.AzureOpenAI(
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_ID")
        )
    else:
        return openai.OpenAI(
            api_key=os.getenv('OPENAI_API_KEY')
        )


In [9]:
openai_client = create_open_ai_client()
openai_model = os.getenv("AZURE_OPENAI_DEPLOYMENT_ID") if IS_AZURE_OPENAI else OPENAI_GPT_MODEL_VERSION

def get_gpt_response(messages, model=openai_model, temperature=0) -> dict:
    response = openai_client.chat.completions.create(
        model=model,
        response_format={"type": "json_object"},  # Uncomment it if your chosen model supports it
        messages=messages,
        temperature=temperature,
    )
    response_str = response.choices[0].message.content

    try:
        response_dict = json.loads(response_str)
    except json.JSONDecodeError:
        print(f"Failed to decode response: {response_str}")
        raise

    return response_dict


In [10]:
print(get_gpt_response([
    {"role": "user", "content": "Say this is test. Format response in JSON"}
]))


{'message': 'test'}


You are all set to use the GPT.

---

## Setup Database

You need a dataset that you will query using natural language. You also need a SQL database that will host that dataset.

### Example Dataset: Digital Music Store

The [Chinook](https://www.sqlitetutorial.net/sqlite-sample-database/) sample database is commonly used for teaching and testing RDBMS concepts. It models a fictitious digital music store with tables for artists, albums, tracks, customers, invoices, and employees. We will use [SQLite](https://www.sqlite.org/index.html) as the database. Python's [sqlite3](https://docs.python.org/3/library/sqlite3.html) package is part of the standard library, so no additional installation or deployment is required.

1. Download the dataset using `curl` or `wget` from [SQLite Tutorial](https://www.sqlitetutorial.net/sqlite-sample-database/).

In [11]:
# Clear previously downloaded and unzipped files

!rm -f ./chinook.zip ./chinook.db

In [12]:
!curl -L0 https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip --output ./chinook.zip

#!wget https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  298k  100  298k    0     0   602k      0 --:--:-- --:--:-- --:--:--  602k


2. Unzip the db file

In [13]:
!unzip chinook.zip


Archive:  chinook.zip
  inflating: chinook.db              


3. The db file will be stored at `./chinook.db`. This is the path you will need when using `sqlite3` package.

In [14]:
!ls -l ./chinook.db


-rw-r--r-- 1 root root 884736 Nov 29  2015 ./chinook.db


4. Extract DB Metadata

In [15]:
import sqlite3


In [16]:
DB_FILE_PATH = "./chinook.db"


In [17]:
def extract_sqlite3_db_metadata(sqlite_db_file_path: str):
    db_metadata = {}

    # Connect to the SQLite database
    conn = sqlite3.connect(sqlite_db_file_path)
    cursor = conn.cursor()

    # Get a list of all tables in the database
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    # Loop through each table and get its columns
    for table in tables:
        table_name = table[0]
        primary_keys = []
        foreign_keys = {}
        columns_info = {}

        # Get table details
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()

        # Extract info about the columns of the current table
        for column in columns:
            column_name = column[1]
            column_type = column[2]
            is_primary = (column[5] == 1)

            columns_info[column_name] = {
                "type": column_type,
                "primary": is_primary,
                "foreign": {}
            }

        # Primary Keys
        primary_keys = [
            c_name
            for c_name, c_attrs in columns_info.items()
            if c_attrs["primary"] == True
        ]

        # Get foreign key details
        cursor.execute(f"PRAGMA foreign_key_list({table_name});")
        fk_constraints = cursor.fetchall()

        for fk in fk_constraints:
            fk_constraint_id = fk[0]
            fk_to_table = fk[2]
            fk_from_column = fk[3]
            fk_to_column = fk[4]

            fk_info = {
                "constraint_id": fk_constraint_id,
                "to_table": fk_to_table,
                "to_column": fk_to_column
            }
            foreign_keys[fk_from_column] = fk_info
            columns_info[fk_from_column]["foreign"] = fk_info

        db_metadata[table_name] = {
            "columns": columns_info,
            "primary_keys": primary_keys,
            "foreign_keys": foreign_keys
        }

    # Close the connection
    conn.close()

    # Remove tables with names staring with "sqlite" as those are not part of applications
    tables_to_remove = [t for t in db_metadata if t.startswith("sqlite")]
    for t in tables_to_remove:
        del db_metadata[t]

    # Done!
    return db_metadata


5. Check out if the database metadata has been extracted correctly.

In [18]:
def table_info_str(t_name, t_info) -> str:
    column_info_str = "\n        ".join([
        f"{c_name}: {c_info['type']}"
        for c_name, c_info in t_info["columns"].items()
    ])

    primary_key_info_str = ""
    if len(t_info["primary_keys"]) > 0:
        primary_key_info_str = f"Primary Keys: {','.join(t_info['primary_keys'])}"

    foreign_key_info_str = ""
    if len(t_info["foreign_keys"]) > 0:
        foreign_key_info_str = "\n    Foreign Keys:\n        " + "\n        ".join([
            f"{fk_from_col} => {fk_info['to_table']}.{fk_info['to_column']}"
            for fk_from_col, fk_info in t_info["foreign_keys"].items()
        ])

    return f"""Table Name: {t_name}
    Columns:
        {column_info_str}
    """ + primary_key_info_str + foreign_key_info_str


In [19]:
chinook_db_metadata = extract_sqlite3_db_metadata(DB_FILE_PATH)


In [20]:
for t_name, t_info in chinook_db_metadata.items():
    print(table_info_str(t_name, t_info))
    print()


Table Name: albums
    Columns:
        AlbumId: INTEGER
        Title: NVARCHAR(160)
        ArtistId: INTEGER
    Primary Keys: AlbumId
    Foreign Keys:
        ArtistId => artists.ArtistId

Table Name: artists
    Columns:
        ArtistId: INTEGER
        Name: NVARCHAR(120)
    Primary Keys: ArtistId

Table Name: customers
    Columns:
        CustomerId: INTEGER
        FirstName: NVARCHAR(40)
        LastName: NVARCHAR(20)
        Company: NVARCHAR(80)
        Address: NVARCHAR(70)
        City: NVARCHAR(40)
        State: NVARCHAR(40)
        Country: NVARCHAR(40)
        PostalCode: NVARCHAR(10)
        Phone: NVARCHAR(24)
        Fax: NVARCHAR(24)
        Email: NVARCHAR(60)
        SupportRepId: INTEGER
    Primary Keys: CustomerId
    Foreign Keys:
        SupportRepId => employees.EmployeeId

Table Name: employees
    Columns:
        EmployeeId: INTEGER
        LastName: NVARCHAR(20)
        FirstName: NVARCHAR(20)
        Title: NVARCHAR(30)
        ReportsTo: INTEGER
 

---

## Database Table Schema Documents

GPT can create a SQL query only if it understands various tables and their columns. While creating the GPT prompt, you must include this info of relevant tables.

The `CREATE TABLE` statement of [SQL DDL](https://en.wikipedia.org/wiki/Data_definition_language) captures all necessary info. Ideally, table description and column descriptions should also be captured as comments to assist document search and GPT.

Let's create a mapping of table name and their `CREATE TABLE` statements.

In [21]:
def create_table_ddl_stmt_str(t_name, t_info) -> str:
    column_defs = ",\n    ".join([
        f"{c_name} \t{c_info['type']}"
        for c_name, c_info in t_info["columns"].items()
    ])

    primary_key_def = ""
    if len(t_info["primary_keys"]) > 0:
        primary_key_def = f",\n\n    PRIMARY KEY ({', '.join(t_info['primary_keys'])})"

    foreign_key_def =""
    if len(t_info["foreign_keys"]) > 0:
        fk_stmts = ",\n".join([
            f"    FOREIGN KEY({fk_from_col}) REFERENCES {fk_info['to_table']}({fk_info['to_column']})"
            for fk_from_col, fk_info in t_info["foreign_keys"].items()
        ])
        foreign_key_def = f",\n\n{fk_stmts}"

    return f"""CREATE TABLE {t_name} (
    {column_defs}{primary_key_def}{foreign_key_def}
);"""


Let's sequence the tables so that the definition of every table referred to in a `FOREIGN KEY` constraint comes before the constraint. While one can write code to analyze foreign key constraint graph and perform a topological sort to get a partial order, I decided to just hand code it as it does not have relevance for this tutorial.

You can check the [Entity Relation Model](https://en.wikipedia.org/wiki/Entity%E2%80%93relationship_model) for all tables drawn using Crow's Foot notation:

<img src="https://www.ml4devs.com/images/illustrations/entity-relationship-diagram-of-chinook-database-schema.webp" alt="Chinook Database Schema" style="width: 1080px; max-width: 100%;">

In [22]:
# Table list in Topological Order for foreign key constraints

chinook_db_table_names = [
    "artists", "albums",
    "media_types", "genres", "tracks",
    "playlists", "playlist_track",
    "employees",
    "customers", "invoices", "invoice_items"
]


In [23]:
all_chinook_db_table_documents: dict[str, str] = {
    t_name: create_table_ddl_stmt_str(t_name, chinook_db_metadata[t_name])
    for t_name in chinook_db_table_names
}


In [24]:
for t_name in chinook_db_table_names:
    print(all_chinook_db_table_documents[t_name])
    print()


CREATE TABLE artists (
    ArtistId 	INTEGER,
    Name 	NVARCHAR(120),

    PRIMARY KEY (ArtistId)
);

CREATE TABLE albums (
    AlbumId 	INTEGER,
    Title 	NVARCHAR(160),
    ArtistId 	INTEGER,

    PRIMARY KEY (AlbumId),

    FOREIGN KEY(ArtistId) REFERENCES artists(ArtistId)
);

CREATE TABLE media_types (
    MediaTypeId 	INTEGER,
    Name 	NVARCHAR(120),

    PRIMARY KEY (MediaTypeId)
);

CREATE TABLE genres (
    GenreId 	INTEGER,
    Name 	NVARCHAR(120),

    PRIMARY KEY (GenreId)
);

CREATE TABLE tracks (
    TrackId 	INTEGER,
    Name 	NVARCHAR(200),
    AlbumId 	INTEGER,
    MediaTypeId 	INTEGER,
    GenreId 	INTEGER,
    Composer 	NVARCHAR(220),
    Milliseconds 	INTEGER,
    Bytes 	INTEGER,
    UnitPrice 	NUMERIC(10,2),

    PRIMARY KEY (TrackId),

    FOREIGN KEY(MediaTypeId) REFERENCES media_types(MediaTypeId),
    FOREIGN KEY(GenreId) REFERENCES genres(GenreId),
    FOREIGN KEY(AlbumId) REFERENCES albums(AlbumId)
);

CREATE TABLE playlists (
    PlaylistId 	INTEGER,
    

---

## Natural Language Query to SQL

With the database set up and all table DDLs generated, we can now build the query pipeline. The approach follows the standard **Retrieval Augmented Generation (RAG)** pattern, adapted for the text-to-SQL problem.

The full pipeline has three phases:

**Embeddings** (data preprocessing)
- Treat each `CREATE TABLE` statement as one document chunk
- Convert each chunk to a vector using an embedding model
- Store vectors in a vector database

**Retrieval** (prompt construction)
- Embed the incoming natural language query using the same model
- Search the vector database for table DDLs with the highest similarity
- Construct a prompt from the query and only the matched table schemas

**Inference** (prompt execution)
- Submit the prompt to the LLM and parse its JSON response
- Execute the returned SQL against the database
- Convert the raw result rows into a plain-English answer

The key motivation for retrieval is **prompt efficiency**: rather than including all 11 table schemas in every prompt (which scales poorly to databases with hundreds of tables), we include only the handful most likely relevant to the user's question.

The following sections implement each phase in order.

### Embeddings: Index Table DDLs in a Vector DB

This is the **Embeddings** phase. Each `CREATE TABLE` statement is treated as one document chunk and indexed in the vector store. When a query arrives later, we search this index to find the most relevant tables instead of sending the full schema to the LLM every time.

**Why DDL statements?** A `CREATE TABLE` statement captures the table name, column names, data types, and foreign key relationships in a compact, structured form that both embedding models and LLMs understand well. It is the natural unit of "what can this table tell us?"

**Embedding model.** We use [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), a 384-dimensional sentence transformer that runs locally. This keeps the notebook fully self-contained, and no API key is needed for embeddings. The model weights (~90 MB) are downloaded once and cached for the session. If retrieval quality is insufficient for your schema, a stronger model such as OpenAI's `text-embedding-3-small` can be swapped in by changing `embedding_fn`.

**Storage.** We use an in-memory [ChromaDB](https://www.trychroma.com/) collection. Nothing is written to disk, so the notebook is safe to re-run from scratch. Each DDL document is stored under its table name as the document ID, making lookup straightforward.

The quick-check cell after upsert confirms that the vector DB returns sensible results: querying "artist and album sales" correctly surfaces the `albums`, `tracks`, and `artists` tables as top matches.

In [25]:
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# In-memory client: nothing is persisted to disk, so the notebook stays re-runnable.
chroma_client = chromadb.EphemeralClient()
table_ddl_collection = chroma_client.get_or_create_collection(
    name="table_ddls",
    embedding_function=embedding_fn,
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
# Index each table's DDL as a chunk; the table name is its document id.
# upsert (instead of add) keeps this cell safe to re-run without duplicate-id errors.
table_ddl_collection.upsert(
    documents=list(all_chinook_db_table_documents.values()),
    ids=list(all_chinook_db_table_documents.keys()),
)

print(f"Indexed {table_ddl_collection.count()} table DDLs in the vector DB")


Indexed 11 table DDLs in the vector DB


In [27]:
# Quick check: which tables does the vector DB return for a sample query?
sample = table_ddl_collection.query(query_texts=["artist and album sales"], n_results=3)
for rank, (t_name, dist) in enumerate(zip(sample["ids"][0], sample["distances"][0]), 1):
    print(f"  {rank}. {t_name}  (distance: {dist:.4f})")


  1. albums  (distance: 0.5702)
  2. tracks  (distance: 0.6504)
  3. artists  (distance: 0.7029)


### Retrieval: Vector DB Document Search

This is the **Retrieval** phase. When the user submits a natural language query, we embed it with the same model used during indexing, search ChromaDB for the closest matching table DDLs, and return only those table schemas.

`retrieve_tables()` wraps this logic: it calls `table_ddl_collection.query()` with the query text and returns a `{table_name: ddl}` dict for the top `TOP_K_TABLES` matches. ChromaDB handles the embedding and nearest-neighbor search internally.

`TOP_K_TABLES = 5` is a tunable threshold. Setting it lower produces a leaner, cheaper prompt but risks omitting a table the query actually needs. Setting it higher improves coverage at the cost of a noisier prompt. For this 11-table database, 5 is a reasonable balance. Feel free to experiment.

Notice that only the retrieved table DDLs are forwarded to the LLM in the next step. This is the central efficiency win of RAG: prompt length grows with the number of *relevant* tables, not with the total schema size.

In [28]:
TOP_K_TABLES = 5  # tune: lower = leaner prompt, higher = more coverage

def retrieve_tables(nl_query: str, top_k: int = TOP_K_TABLES) -> dict[str, str]:
    # Embed nl_query with the same model and search the vector DB for the
    # most similar table DDL chunks, then return {table_name: ddl} for the top matches.
    results = table_ddl_collection.query(query_texts=[nl_query], n_results=top_k)
    retrieved_table_names = results["ids"][0]
    return {
        t_name: all_chinook_db_table_documents[t_name]
        for t_name in retrieved_table_names
    }


### Prompt Construction

With the relevant table schemas retrieved, we construct a two-part prompt:

**System prompt** (`nl2sql_system_prompt`): Provides the LLM with its role ("data analyst and SQL expert"), the retrieved `CREATE TABLE` statements, and step-by-step instructions: identify relevant tables, identify relevant columns, then craft an optimal SQL query. It also specifies the exact JSON output format: a `tables` key listing which columns were used, and an `sql` key with the query string. Requesting structured JSON output makes the response easy to parse and chain into the execution step.

**User prompt** (`nl2sql_user_prompt`): Wraps the user's natural language question in triple backticks and asks the LLM to write the corresponding SQL.

Separating system and user turns follows the standard chat-model convention: the system turn sets persistent context and constraints; the user turn carries the per-request input. This separation also makes it easy to reuse the system prompt across multiple user queries without reconstructing it each time.

A few things worth noting about the prompt design:
- The DDL schemas are labeled with the table name so the LLM can refer back to them unambiguously.
- The instructions ask for an "optimal order" of SELECT, filter, GROUP, and JOIN, therefore nudging the model to think about query structure rather than just copying column names.
- `sql_flavor` defaults to "Python sqlite3," which signals to the LLM which SQL dialect and functions are available.

In [29]:
def nl2sql_system_prompt(documents: dict[str, str], sql_flavor: str = "Python sqlite3") -> str:
    metadata = "\n".join([
        f"# SQL DDL Schema for `{table_name}` table:```sql\n{table_schema}```\n"
        for table_name, table_schema in documents.items()
    ])

    system_prompt = f"""
    You are a data analyst and data engineer. You are an expert in writing SQL queries
    for {sql_flavor} database.

    You have following tables in the database. The table name is in single backquote, and
    the DDL code to create that table with schema and metadata details are in triple backquote.

    ### Database Table Schemas:
    \n{metadata}
    ###

    User ask you queries in natural language, and you job is to write equivalent
    SQL queries in following steps:
    1. Identify the tables that have data relevant for the query
    2. Identify relevant columns in those tables
    3. Craft a SQL query that selects, filters, groups, joins in an optimal order
       that is equivalent to the user's natural language query.

    Format your response as a JSON dictionary (only parsable JSON, with no text or explanation surrounding it) with following key, value:
    - tables: a dictionary with the name of relevant tables as keys, and the
        list of relevant columns in that as value.
    - sql: the sql query that you crafted.
    """

    return system_prompt


In [30]:
def nl2sql_user_prompt(nl_query: str):
    return f"Write a SQL that computes natural language query in triple backquotes: ```{nl_query}```"


### Prompt Execution

`write_sql_query()` ties the retrieval and prompt steps together: it calls `retrieve_tables()` to get the most relevant schemas, builds the system and user prompts, sends both to the LLM via `get_gpt_response()`, and returns the parsed JSON response.

The response dict contains two keys:
- `tables`: which tables and columns the LLM identified as relevant to the question
- `sql`: the SQL query it crafted

At this point we have a SQL string but no results yet. The next step executes it against the database.

In [31]:
def write_sql_query(nl_query: str) -> dict:
    # Vectorize nl_query and find matching documents (tables and their DDL)
    documents = retrieve_tables(nl_query)
    # Craft prompt using the natural language queries and matching documents
    system_prompt = nl2sql_system_prompt(documents)
    user_prompt = nl2sql_user_prompt(nl_query)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    response_dict = get_gpt_response(messages)

    return response_dict


### Post-processing: Execute the SQL

The SQL string returned by the LLM is executed against the Chinook SQLite database using SQLAlchemy. Two implementations are provided:

- `execute_sql_query_on_sqlite3()`: Uses the standard library `sqlite3` module directly. Simple, but left as a commented-out fallback.
- `execute_sql_query()`: Uses a SQLAlchemy `Connection`, which adds engine-level logging (visible in the cell outputs below) and makes it easy to swap SQLite for any other database by changing the connection URL string.

`sql_engine` is created with `echo=True` so every executed statement is printed. This is helpful during development for verifying that the LLM-generated SQL is syntactically valid and reads the way you expect before trusting the results.

In a production system, this step would also include validation: catching SQL syntax errors, handling empty result sets gracefully, and potentially rejecting queries that touch write operations.

In [32]:
def execute_sql_query_on_sqlite3(sql_query: str):
    conn = sqlite3.connect(DB_FILE_PATH)
    cursor = conn.cursor()
    result = cursor.execute(sql_query)
    rows = result.fetchall()
    conn.close()

    return rows


In [33]:
import sqlalchemy


In [34]:
sql_engine = sqlalchemy.create_engine(
    f"sqlite:///{os.path.abspath(os.path.join(os.getcwd(), DB_FILE_PATH))}",
    echo=True
)


In [35]:
def execute_sql_query(connection, query):
    result_obj = connection.execute(sqlalchemy.text(query))
    return result_obj.fetchall()


### Post-processing: Natural Language Answer

Raw database rows work for a developer, but an end user expects a sentence, not a list of tuples. This step closes the loop: `generate_nl_answer()` sends the user's original question, the SQL that answered it, and the result rows back to the LLM and asks for a short, plain-English response.

`nl_answer_prompt()` constructs this prompt carefully:
- It provides all three inputs (question, SQL, rows) as context so the answer is grounded in real data.
- It explicitly instructs the LLM not to mention SQL, table names, or column names in the answer. The goal is a natural response, not a technical summary.
- It handles the empty-result case: "if there are no rows, say that no matching data was found."
- It requests a single-key JSON response (`answer`) for easy extraction.

This pattern of using the LLM both to generate SQL and to narrate results is sometimes called **generate-then-explain**. It keeps the final response accurate (it references only the rows returned) while keeping the user experience conversational.

In [36]:
def nl_answer_prompt(nl_query: str, sql_query: str, rows: list) -> str:
    return f"""A user asked the following question in natural language:
```{nl_query}```

It was answered by running this SQL query:
```sql
{sql_query}
```

The query returned these rows (a list of tuples):
```{rows}```

Write a short, friendly answer to the user's question in plain English, using the
returned rows. Do not mention SQL, tables, or column names. If there are no rows,
say that no matching data was found.

Format your response as a JSON dictionary with a single key:
- answer: your natural language answer as a string.
"""

def generate_nl_answer(nl_query: str, sql_query: str, rows: list) -> str:
    messages = [
        {"role": "user", "content": nl_answer_prompt(nl_query, sql_query, rows)}
    ]
    response_dict = get_gpt_response(messages)
    return response_dict["answer"]


In [37]:
def execute_nl_query(nl_query: str):
    response = write_sql_query(nl_query)

    #response["rows"] = execute_sql_query_on_sqlite3(response["sql"])
    with sql_engine.connect() as conn:
        response["rows"] = execute_sql_query(conn, response["sql"])

    # Turn the raw rows into a human-friendly natural language answer
    response["answer"] = generate_nl_answer(nl_query, response["sql"], response["rows"])

    return response


### Try It Out

With all pieces in place, `execute_nl_query()` is the single entry point: it accepts a plain-English question, retrieves relevant tables, generates SQL, executes it, and returns the SQL, the result rows, and a natural language answer in one response dict.

Three representative queries are run to test the pipeline end-to-end:

1. **"Who is the artist with the most albums?"**  
   A simple aggregation with a JOIN between `artists` and `albums`.
2. **"List the top 3 tracks with maximum sale."**  
   A ranking query joining `tracks` and `invoice_items`, with a computed sum.
3. **"Name the employee who supports maximum number of customers."**  
   A JOIN between `employees` and `customers` with a COUNT and ORDER BY.

These cover the most common SQL patterns (GROUP BY, ORDER BY, LIMIT, JOIN) and exercise different parts of the schema. For each query, inspect:
- `tables`: which tables and columns the LLM decided were relevant
- `sql`: the exact SQL it generated
- `rows`: the raw result tuples from the database
- `answer`: the plain-English response the LLM composed from those rows

In [38]:
test_nl_queries = [
    "Who is the artist with the most albums?",
    "List the top 3 tracks with maximum sale.",
    "Name the employee who supports maximum number of customers."
]


In [39]:
results = []
for nl_q in test_nl_queries:
    response =  execute_nl_query(nl_q)
    response["query"] = nl_q
    results.append(response)


2026-06-23 18:28:37,557 INFO sqlalchemy.engine.Engine BEGIN (implicit)


INFO:sqlalchemy.engine.Engine:BEGIN (implicit)


2026-06-23 18:28:37,560 INFO sqlalchemy.engine.Engine SELECT ar.Name AS ArtistName,
       COUNT(a.AlbumId) AS AlbumCount
FROM artists ar
JOIN albums a
  ON a.ArtistId = ar.ArtistId
GROUP BY ar.ArtistId, ar.Name
ORDER BY AlbumCount DESC
LIMIT 1;


INFO:sqlalchemy.engine.Engine:SELECT ar.Name AS ArtistName,
       COUNT(a.AlbumId) AS AlbumCount
FROM artists ar
JOIN albums a
  ON a.ArtistId = ar.ArtistId
GROUP BY ar.ArtistId, ar.Name
ORDER BY AlbumCount DESC
LIMIT 1;


2026-06-23 18:28:37,562 INFO sqlalchemy.engine.Engine [generated in 0.00469s] ()


...

INFO:sqlalchemy.engine.Engine:SELECT e.EmployeeId,
       e.FirstName,
       e.LastName
FROM employees e
JOIN customers c
  ON c.SupportRepId = e.EmployeeId
GROUP BY e.EmployeeId, e.FirstName, e.LastName
ORDER BY COUNT(*) DESC
LIMIT 1;


2026-06-23 18:28:44,090 INFO sqlalchemy.engine.Engine [generated in 0.00395s] ()


INFO:sqlalchemy.engine.Engine:[generated in 0.00395s] ()


2026-06-23 18:28:44,091 INFO sqlalchemy.engine.Engine ROLLBACK


INFO:sqlalchemy.engine.Engine:ROLLBACK


In [40]:
for t in results:
    print(
        f"""User Query: {t['query']}
        Answer:
            {t['answer']}
        Tables:
            {json.dumps(t['tables'])}
        SQL:
            {t['sql']}
        Rows:
            {str(t['rows'])}\n
        """)


User Query: Who is the artist with the most albums?
        Answer:
            Iron Maiden has the most albums, with 21.
        Tables:
            {"albums": ["AlbumId", "ArtistId"], "artists": ["ArtistId", "Name"]}
        SQL:
            SELECT ar.Name AS ArtistName,
       COUNT(a.AlbumId) AS AlbumCount
FROM artists ar
JOIN albums a
  ON a.ArtistId = ar.ArtistId
GROUP BY ar.ArtistId, ar.Name
ORDER BY AlbumCount DESC
LIMIT 1;
        Rows:
            [('Iron Maiden', 21)]

        
User Query: List the top 3 tracks with maximum sale.
        Answer:
            Here are the top 3 tracks by total sales: 1) “The Woman King” — 3.98 2) “The Fix” — 3.98 3) “Walkabout” — 3.98.
        Tables:
            {"tracks": ["TrackId", "Name"], "invoice_items": ["TrackId", "UnitPrice", "Quantity"]}
        SQL:
            SELECT
  t.TrackId,
  t.Name,
  SUM(ii.UnitPrice * ii.Quantity) AS TotalSales
FROM invoice_items ii
JOIN tracks t
  ON t.TrackId = ii.TrackId
GROUP BY t.TrackId, t.Name
ORDE

## Cleanup

Dispose of the SQLAlchemy engine to release the database connection pool before the notebook session ends.

In [41]:
sql_engine.dispose()


---

## Summary and Key Takeaways

This tutorial built a working natural language to SQL pipeline from scratch, demonstrating how to combine a vector store, an embedding model, and an LLM to let users query a relational database in plain English.

### What We Built

| Step | What it does |
|---|---|
| Schema extraction | Reads table metadata from SQLite using `PRAGMA` calls and converts it to DDL strings |
| Embeddings | Encodes each table DDL as a 384-dim vector using `all-MiniLM-L6-v2` and stores it in ChromaDB |
| Retrieval | Embeds the user query and returns the top-K most similar table DDLs |
| SQL generation | Sends retrieved schemas and the user query to an LLM, which returns structured JSON with `tables` and `sql` |
| Execution | Runs the SQL against SQLite via SQLAlchemy and fetches result rows |
| Natural language answer | Sends the question, SQL, and rows back to the LLM to produce a plain-English response |

### Key Concepts Covered

- **RAG for structured data**: The same retrieve-then-generate pattern used for document Q&A applies to database queries. The "documents" are just table DDL statements.
- **DDL as a retrieval unit**: A `CREATE TABLE` statement is a compact, information-rich representation of a table that works well for both semantic search and LLM context.
- **Prompt design for SQL**: System/user turn separation, structured JSON output format, and explicit step-by-step instructions all improve the reliability of LLM-generated SQL.
- **Generate-then-explain**: Using the LLM a second time to narrate query results keeps the final answer grounded in real data while keeping the user experience conversational.

### Where to Go Next

- **Improve retrieval**: Try hybrid search (BM25 + dense) or add a re-ranker to improve table selection on larger schemas.
- **Add schema annotations**: Enrich the DDL chunks with natural-language column descriptions and sample values so the embedding model has more signal to work with.
- **Handle errors gracefully**: Wrap SQL execution in a retry loop that sends failed queries back to the LLM for self-correction.
- **Extend to other databases**: Swap `sqlite:///` for a PostgreSQL or MySQL connection URL; the rest of the pipeline is dialect-agnostic.
- **Explore agent-based approaches**: For multi-step queries or queries that require clarification, an LLM agent with tool use can iterate over running SQL, checking results, and refining the query, rather than making a single generation pass.

---
<p>Copyright &copy 2023 - 2026 <a href="https://www.linkedin.com/in/scgupta">Satish Chandra Gupta</a>.</p>
<img src="https://licensebuttons.net/l/by-nc-sa/3.0/88x31.png" align="left"/> <p>&nbsp;<a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">CC BY-NC-SA 4.0 International</a> License.</p>